In [45]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
df = pd.read_csv("../data/train.csv")

X=df.drop("Survived", axis=1)
y=df["Survived"]
X_train, X_val, y_train, y_val=train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)
X_train.shape, X_val.shape
df["CabinKnown"] = df["Cabin"].notna().astype(int)
df["Title"] = df["Name"].str.extract(r",\s*([^.]*)\.")
df[["Name", "Title"]].head()
title_counts = df["Title"].value_counts()

rare_titles = title_counts[title_counts < 10].index

df["TitleGrouped"] = df["Title"].replace(rare_titles, "Rare")
df["TitleGrouped"].value_counts()
df["Age"].isnull().sum()
features = [
    "Pclass",
    "Sex",
    "Age",
    "SibSp",
    "Parch",
    "Fare",
    "CabinKnown",
    "Embarked",
    "TitleGrouped"
]

X = df[features]
y = df["Survived"]
X.head()
numeric_features = [
    "Pclass",
    "Age",
    "SibSp",
    "Parch",
    "Fare",
    "CabinKnown"
]
categorical_features = [
    "Sex",
    "Embarked",
    "TitleGrouped"
]
preprocessor = ColumnTransformer(
    transformers=[
        ("num", "passthrough", numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ]
)
model = DecisionTreeClassifier(random_state=42)
model_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", model)
    ]
)
X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)
X_train.shape, X_val.shape
from sklearn.impute import SimpleImputer
numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median"))
    ]
)
model_pipeline.fit(X_train, y_train)
y_pred = model_pipeline.predict(X_val)
accuracy = accuracy_score(y_val, y_pred)
accuracy
cm = confusion_matrix(y_val, y_pred)
cm
print(classification_report(y_val, y_pred))
from sklearn.model_selection import cross_val_score
cv_scores = cross_val_score(
    model_pipeline,
    X_train,
    y_train,
    cv=5,
    scoring="accuracy"
)
cv_scores

cv_scores.std()
cv_scores.mean()
depths = [2, 3, 4, 5, 6, 8, 10]
depth_scores = {}

for depth in depths:
    tree = DecisionTreeClassifier(
        max_depth=depth,
        random_state=42
    )

    pipeline = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", tree)
        ]
    )

    scores = cross_val_score(
        pipeline,
        X_train,
        y_train,
        cv=5,
        scoring="accuracy"
    )

    depth_scores[depth] = scores.mean()
    depth_scores

              precision    recall  f1-score   support

           0       0.84      0.84      0.84       110
           1       0.74      0.75      0.75        69

    accuracy                           0.80       179
   macro avg       0.79      0.79      0.79       179
weighted avg       0.81      0.80      0.80       179

